In [1]:
import pandas as pd
import os
import pickle
from sklearn.metrics import mean_absolute_error, accuracy_score, mean_absolute_percentage_error
import numpy as np
os.chdir("../../data")

In [2]:
dataset = "HANCOCK"
imputed_dir = "imputation_data"
groundtruth_dir = f'imputation_groundtruth/{dataset}'
bins = 7
discretization = "nonlinear"

In [3]:
def save_result_to_dict(results_dict : dict,
                        num_run : int,
                        na_ratio: float,
                        tool_name : str,
                        mae_cont : float,
                        acc_cat : float,
                        mre_cont: float,
                        dim : int = -1,
                        bins : int = -1,
                        discretization : str = ""
                        ):
    results_dict['run'].append(num_run)
    results_dict['na_ratio'].append(na_ratio)
    results_dict['tool'].append(tool_name)
    results_dict['mae_cont'].append(mae_cont)
    results_dict['acc_cat'].append(acc_cat)
    results_dict['mre_cont'].append(mre_cont)
    results_dict["dim"].append(dim)
    results_dict["bins"].append(bins)
    results_dict["discretization"].append(discretization)
    return results_dict

In [ ]:
imputation_error_dict = {'run' : [], 'na_ratio' : [], 'tool': [], 'mae_cont': [], 'acc_cat' : [], 'mre_cont': [], 'dim': [], "bins": [], "discretization": []}
# Iterate over all groundtruth files and look up corresponding imputed values.
for gt_file in os.listdir(groundtruth_dir):
    
    gt_file_path = os.path.join(groundtruth_dir, gt_file)
    with open(gt_file_path, 'rb') as f:
        gt_dict = pickle.load(f)
    
    print("Processing ", gt_file_path)
    # Remove the .pkl extension and change last value to float representation.
    base = gt_file.rsplit('.', 1)[0]
    parts = base.split('_')
    parts[-1] = str(float(parts[-1]))
    # Join the parts back and add the new extension
    imputation_file = '_'.join(parts) + '.tsv'
    imputation_file_ac = '_'.join(parts) + '.csv'
    
    # Open KNN imputed file.
    knn_file = os.path.join(imputed_dir, 'knn', dataset, 'imputed_data', imputation_file)
    # Structure: index are samples, columns are variables.
    knn_df = pd.read_csv(knn_file, sep='\t', index_col=0)
    
    # Open missforest imputed file.
    miss_file = os.path.join(imputed_dir, 'missforest', dataset, 'imputed_data', imputation_file)
    # Structure: index are samples, columns are variables.
    miss_df = pd.read_csv(miss_file, sep='\t', index_col=0)
    
    # Open AC imputed file.
    ac_file = os.path.join(imputed_dir, 'autocomplete', dataset, 'imputed_data', imputation_file_ac)
    # Structure: index are samples, columns are variables.
    ac_df = pd.read_csv(ac_file)
    ac_df.set_index('ID', inplace=True)
    
    # Open graph-based imputed file.
    graph_file_16 = os.path.join(imputed_dir, 'pome_based', dataset, f'imputed_{discretization}_{bins}_16', imputation_file)
    if not os.path.exists(graph_file_16):
        continue
    # Structure: index are variables, columns are samples.
    graph_df_16 = pd.read_csv(graph_file_16, sep='\t', index_col=0)
    
    graph_file_32 = os.path.join(imputed_dir, 'pome_based', dataset, f'imputed_{discretization}_{bins}_32', imputation_file)
    if not os.path.exists(graph_file_32):
        continue
    # Structure: index are variables, columns are samples.
    graph_df_32 = pd.read_csv(graph_file_32, sep='\t', index_col=0)
    
    graph_file_64 = os.path.join(imputed_dir, 'pome_based', dataset, f'imputed_{discretization}_{bins}_64', imputation_file)
    if not os.path.exists(graph_file_64):
        continue
    # Structure: index are variables, columns are samples.
    graph_df_64 = pd.read_csv(graph_file_64, sep='\t', index_col=0)
    
    # Iterate over all copy-masked data entries.
    gt_cont = []
    gt_cat = []
    graph_cont_16 = []
    graph_cat_16 = []
    graph_cont_32 = []
    graph_cat_32 = []
    graph_cont_64 = []
    graph_cat_64 = []
    knn_cont = []
    knn_cat = []
    ac_cont = []
    ac_cat = []
    missf_cont = []
    missf_cat = []
    for pos, value in gt_dict.items():
        sample = pos[0]
        variable = pos[1]
        gt_value = value[0]
        gt_type = value[1]
        
        # Read corresponding imputed values.
        knn_value = knn_df.loc[sample, variable]
        miss_value = miss_df.loc[sample, variable]
        ac_value = ac_df.loc[sample, variable]
        graph_value_16 = graph_df_16.loc[variable, sample]
        graph_value_32 = graph_df_32.loc[variable, sample]
        graph_value_64 = graph_df_64.loc[variable, sample]
        # Compute error metrics.
        if gt_type == 'cont':
            gt_cont.append(gt_value)
            knn_cont.append(knn_value)
            ac_cont.append(ac_value)
            missf_cont.append(miss_value)
            graph_cont_16.append(graph_value_16)
            graph_cont_32.append(graph_value_32)
            graph_cont_64.append(graph_value_64)
            
        else:
            gt_cat.append(gt_value)
            knn_cat.append(np.round(knn_value))
            ac_cat.append(np.round(ac_value))
            missf_cat.append(np.round(miss_value))
            graph_cat_16.append(graph_value_16)
            graph_cat_32.append(graph_value_32)
            graph_cat_64.append(graph_value_64)
            

    ac_cont_array = np.array(ac_cont)
    gt_cont_array = np.array(gt_cont)
    ac_cat_array = np.array(ac_cat)
    mask = ~np.isnan(ac_cont_array) & ~np.isnan(gt_cont_array)
    max_difference = np.max(np.abs(ac_cont_array[mask] - gt_cont_array[mask]))
    # Check if AC arrays contain NAs and update accordingly.
    if np.isnan(ac_cont_array).sum() > 0 or np.isnan(ac_cat_array).sum() > 0:
        ac_cont_array[np.isnan(ac_cont_array)] = gt_cont_array[np.isnan(ac_cont_array)] + max_difference
        ac_cat_array[np.isnan(ac_cat_array)] = -1
        ac_cont = list(ac_cont_array)
        ac_cat = list(ac_cat_array)
    
    # Imputation setup specfication.
    NUM_RUN = parts[-1]
    NA_RATIO = parts[2]
    gt_joint = gt_cont + gt_cat
    
    # Compute KNN imputation errors.
    knn_mae_cont = mean_absolute_error(gt_cont, knn_cont)
    knn_mre_cont = mean_absolute_percentage_error(gt_cont, knn_cont)
    knn_acc = accuracy_score(gt_cat, knn_cat)
    imputation_error_dict = save_result_to_dict(
        results_dict=imputation_error_dict,
        num_run=NUM_RUN,
        na_ratio=NA_RATIO,
        tool_name="KNN",
        mae_cont=knn_mae_cont,
        acc_cat=knn_acc,
        mre_cont=knn_mre_cont
    )
    
    # Compute AC imputation errors.
    ac_mae_cont = mean_absolute_error(gt_cont, ac_cont)
    ac_mre_cont = mean_absolute_percentage_error(gt_cont, ac_cont)
    ac_acc = accuracy_score(gt_cat, ac_cat)
    imputation_error_dict = save_result_to_dict(
        results_dict=imputation_error_dict,
        num_run=NUM_RUN,
        na_ratio=NA_RATIO,
        tool_name="AutoComplete",
        mae_cont=ac_mae_cont,
        acc_cat=ac_acc,
        mre_cont=ac_mre_cont
    )
    
    # Compute missForest imputation errors.
    miss_mae_cont = mean_absolute_error(gt_cont, missf_cont)
    miss_mre_cont = mean_absolute_percentage_error(gt_cont, missf_cont)
    miss_acc = accuracy_score(gt_cat, missf_cat)
    imputation_error_dict = save_result_to_dict(
        results_dict=imputation_error_dict,
        num_run=NUM_RUN,
        na_ratio=NA_RATIO,
        tool_name="MissForest",
        mae_cont=miss_mae_cont,
        acc_cat=miss_acc,
        mre_cont=miss_mre_cont
    )
    
    # Compute graph-based imputation errors.
    graph_mae_cont_16 = mean_absolute_error(gt_cont, graph_cont_16)
    graph_mre_cont_16 = mean_absolute_percentage_error(gt_cont, graph_cont_16)
    graph_acc_16 = accuracy_score(gt_cat, graph_cat_16)
    imputation_error_dict = save_result_to_dict(
        results_dict=imputation_error_dict,
        num_run=NUM_RUN,
        na_ratio=NA_RATIO,
        tool_name="POME",
        mae_cont=graph_mae_cont_16,
        acc_cat=graph_acc_16,
        mre_cont=graph_mre_cont_16,
        dim=16,
        bins=bins,
        discretization=discretization
    )
    
    # Compute graph-based imputation errors.
    graph_mae_cont_32 = mean_absolute_error(gt_cont, graph_cont_32)
    graph_mre_cont_32 = mean_absolute_percentage_error(gt_cont, graph_cont_32)
    graph_acc_32 = accuracy_score(gt_cat, graph_cat_32)
    imputation_error_dict = save_result_to_dict(
        results_dict=imputation_error_dict,
        num_run=NUM_RUN,
        na_ratio=NA_RATIO,
        tool_name="POME",
        mae_cont=graph_mae_cont_32,
        acc_cat=graph_acc_32,
        mre_cont=graph_mre_cont_32,
        dim=32,
        bins=bins,
        discretization=discretization
    )    
    
    # Compute graph-based imputation errors.
    graph_mae_cont_64 = mean_absolute_error(gt_cont, graph_cont_64)
    graph_mre_cont_64 = mean_absolute_percentage_error(gt_cont, graph_cont_64)
    graph_acc_64 = accuracy_score(gt_cat, graph_cat_64)
    imputation_error_dict = save_result_to_dict(
        results_dict=imputation_error_dict,
        num_run=NUM_RUN,
        na_ratio=NA_RATIO,
        tool_name="POME",
        mae_cont=graph_mae_cont_64,
        acc_cat=graph_acc_64,
        mre_cont=graph_mre_cont_64,
        dim=64,
        bins=bins,
        discretization=discretization
    )    
        
    

In [ ]:
# Transform results into dataframe structure.
results_df = pd.DataFrame(imputation_error_dict)
print(results_df)
#results_df.to_csv(f'{dataset}_imputation_{discretization}_bins_{bins}.csv', index=True)